### Installing Neccessary Libraries

In [1]:
!pip install tqdm
!pip install transformers==4.21.0
!pip install python-terrier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 306.2 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.0
    Uninstalling tokenizers-0.22.0:
      Successfully uninstalled tokenizers-0.22.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [tokenizers]  WARNING: Failed to remove contents in a temporary directory '/mnt/primary/conda-envs/colbertv2/lib/python3.9/site-packages/~.kenizers'.
  You can safely remove it manually.
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.12m0/2 [tokenizers]
    Uninstalling transformers-4.56.1:╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-4.56.1━━━━━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages th

### Importing neccessary libraries

In [2]:
import pandas as pd
import torch
from tqdm import tqdm
import ir_datasets
import pyterrier as pt
from transformers import AutoModelForMaskedLM, AutoTokenizer
from scipy.sparse import csr_matrix

/opt/miniconda3/envs/colbertv2/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Initialize PyTerrier
if not pt.started():
    pt.java.init()

/tmp/ipykernel_16977/1950713485.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


### Load and Prepare dataset

In [4]:
# Load SARA dataset
dataset = ir_datasets.load("sara")
documents = [doc.text for doc in dataset.docs_iter()]
doc_ids = [doc.doc_id for doc in dataset.docs_iter()]
idx_to_doc_id = {i: doc_id for i, doc_id in enumerate(doc_ids)}

# Extract queries and create topics dataframe
queries = [query.text for query in dataset.queries_iter()]
query_ids = [query.query_id for query in dataset.queries_iter()]
topics = pd.DataFrame({'qid': query_ids, 'query': queries})

# Extract relevance judgments and format it for evaluation
qrels_list = list(dataset.qrels_iter())
qrels = pd.DataFrame(qrels_list)
qrels.rename(columns={'query_id': 'qid', 'doc_id': 'docno', 'label': 'label'}, inplace=True)
qrels['docno'] = qrels['docno'].astype(str)

### Load Splade Model

In [5]:
# Load SPLADEv2 model and tokenizer
splade_model_name = "naver/splade-cocondenser-ensembledistil"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(splade_model_name)
model = AutoModelForMaskedLM.from_pretrained(splade_model_name).to(device)

# Set to evaluation mode
model.eval()

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

### Document Encoding and Retrieval Functions

In [6]:
def encode_splade(text, tokenizer, model, max_length=512):

    # Tokenize input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       padding=True, max_length=max_length).to(device)
    
    # Generate SPLADE representation without gradient computation
    with torch.no_grad():
        
        logits = model(**inputs).logits
        # SPLADE log(1 + relu(x)) sparse encoding
        sparse_rep = torch.log(1 + torch.relu(logits)) * inputs.attention_mask.unsqueeze(-1)
        sparse_rep = torch.max(sparse_rep, dim=1)[0]  # max pooling over sequence length
        
    return sparse_rep.cpu().numpy()


def compute_similarity(query_rep, doc_rep):

    # Dot product similarity between query and document SPLADE representations
    return float(csr_matrix(query_rep).dot(csr_matrix(doc_rep).T).toarray()[0, 0])

# Retrieve top-k documents for each query
def retrieve_splade(topics, doc_embs, idx_to_doc_id, k=100):
    
    results = []

    # Process each query
    for _, row in topics.iterrows():
        qid = row['qid']
        query = row['query']
        query_rep = encode_splade(query, tokenizer, model)

        # Calculate similarity scores with all documents
        scores = [compute_similarity(query_rep, doc_rep) for doc_rep in doc_embs]

        # Get top-k document indices by score
        topk_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]

        # Build results for this query
        for rank, idx in enumerate(topk_idx):
            results.append({
                'qid': qid,
                'docno': str(idx_to_doc_id[idx]),
                'score': scores[idx],
                'rank': rank
            })
            
    return pd.DataFrame(results)

In [8]:
# Precompute document embeddings for all documents
doc_embs = [encode_splade(doc, tokenizer, model) for doc in documents]
print("Document encoding with SPLADEv2 completed")

# Perform retrieval using SPLADEv2
print("Running SPLADEv2 retrieval on SARA dataset")
scores = retrieve_splade(topics, doc_embs, idx_to_doc_id, k=100)
print(f"Retrieved {len(scores)} results")

# Ready for evaluation
scores['docno'] = scores['docno'].astype(str)
qrels['docno'] = qrels['docno'].astype(str)

Document encoding with SPLADEv2 completed
Running SPLADEv2 retrieval on SARA dataset
Retrieved 15000 results


In [9]:
scores.head()

,qid,docno,score,rank
0,1,191480,6.876578,0
1,1,135767,6.153030,1
2,1,120514,6.148225,2
3,1,175162,6.029330,3
4,1,6178,5.775966,4


### Evaluation

In [14]:
# Evaluate with PyTerrier
metrics = ['ndcg_cut_10', 'map', 'P_10', 'recall_10']
eval_results = pt.Evaluate(scores, qrels, metrics=metrics)

print("Evaluation Completed")

Evaluation Completed


In [15]:
results = pd.DataFrame(eval_results, index=[0])
results = results.round(4)

# Add model column
results.insert(0, "Model", "Splade")

In [16]:
results

,Model,ndcg_cut_10,map,P_10,recall_10
0,Splade,0.173,0.1251,0.1553,0.1502


In [ ]:
# Save the results
results.to_csv("retrieval_scores/splade_scores.csv")